# prepare_splits v3: estratificacion a nivel paciente

**Bug corregido en v3:**

En v2 pasamos TODAS las imagenes al estratificador (con labels individuales).
El problema: un paciente positivo puede tener 5 imagenes "No Finding"
de visitas diferentes. Eso confunde al algoritmo y rompe la estratificacion.

**Solucion v3:**

1. Colapsar a un row por paciente (`patient_label = max(labels)`).
2. Estratificar a NIVEL PACIENTE.
3. Expandir de vuelta a todas las imagenes de cada paciente.

Tambien se corrige el balanceo: ahora submuestrea PACIENTES negativos,
no imagenes negativas (mantiene la integridad del split por paciente).

**Resultado esperado:** las tres proporciones dentro de ~5pp del global.

**Runtime:** CPU.


## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'
print(f'CODE_DIR: {CODE_DIR}')


## 2. Sobrescribir `scripts/prepare_splits.py` con v3

In [ ]:
from pathlib import Path

script_content = r'''"""
scripts/prepare_splits.py  (v3)

Genera los splits train/val/test para el baseline binario del Sprint 1.

CAMBIO CLAVE frente a v2: estratificacion a NIVEL PACIENTE, no a nivel imagen.

Problema de la v2:
  StratifiedGroupKFold recibia todas las imagenes con sus labels individuales,
  pero un paciente positivo puede tener varias imagenes "No Finding"
  (visitas diferentes). Esto confundia al estratificador y producia splits
  desbalanceados (ejemplo real: test quedo con 12% positivos vs 33% global).

Solucion v3:
  1. Colapsar el dataset a un row por paciente, con patient_label = max(labels).
     Justificacion: si un paciente tuvo cardiomegalia alguna vez, el paciente
     "tiene la condicion" aunque no se vea en todas sus radiografias.
  2. Correr StratifiedKFold sobre pacientes (ya no necesitamos Grouped porque
     cada paciente es un unico row).
  3. Expandir el resultado a todas las imagenes de cada paciente.
  4. Validar proporciones y ausencia de leakage.

Resultado esperado con 367 pacientes (93 pos, 274 neg):
  Las tres proporciones de positivos deben quedar dentro de 5pp del global.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.training.utils import set_seed, get_logger, load_config  # noqa: E402


# ---------------------------------------------------------------------------
# Construccion de labels
# ---------------------------------------------------------------------------
def build_binary_labels(df: pd.DataFrame, positive_class: str,
                        negative_class: str) -> pd.DataFrame:
    df = df.copy()
    is_positive = df['Finding Labels'].str.contains(positive_class, regex=False)
    is_negative = df['Finding Labels'] == negative_class
    df['label'] = np.where(is_positive, 1,
                   np.where(is_negative, 0, np.nan))
    return df


def filter_views(df: pd.DataFrame, views: list[str]) -> pd.DataFrame:
    return df[df['View Position'].isin(views)].copy()


def undersample_negatives(df: pd.DataFrame, ratio: float,
                          seed: int) -> pd.DataFrame:
    """
    Submuestrea negativos a nivel PACIENTE para respetar el ratio.

    Nota: submuestreamos pacientes negativos, no imagenes negativas. Asi
    mantenemos la integridad del split por paciente.
    """
    positives = df[df['label'] == 1]
    negatives = df[df['label'] == 0]

    # Contamos pacientes unicos de cada clase
    pos_patients = positives['Patient ID'].unique()
    neg_patients = negatives['Patient ID'].unique()

    # Eliminar pacientes negativos que tambien aparecen como positivos
    # (un paciente que tiene alguna imagen positiva NO es "negativo puro")
    neg_patients_clean = np.setdiff1d(neg_patients, pos_patients)

    n_pos_pat = len(pos_patients)
    n_neg_pat_target = min(len(neg_patients_clean), int(n_pos_pat * ratio))

    rng = np.random.default_rng(seed)
    rng.shuffle(neg_patients_clean)
    neg_patients_sampled = neg_patients_clean[:n_neg_pat_target]

    keep_patients = set(pos_patients) | set(neg_patients_sampled)
    balanced = df[df['Patient ID'].isin(keep_patients)].copy()
    balanced = balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return balanced


# ---------------------------------------------------------------------------
# Stratified Split a nivel paciente (v3)
# ---------------------------------------------------------------------------
def collapse_to_patient_level(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reduce el DataFrame a un row por paciente.
    patient_label = 1 si el paciente tiene al menos una imagen positiva.
    """
    patient_df = (df.groupby('Patient ID')['label']
                    .max()
                    .astype(int)
                    .reset_index()
                    .rename(columns={'label': 'patient_label'}))
    return patient_df


def stratified_patient_split(df: pd.DataFrame, train_ratio: float,
                             val_ratio: float, test_ratio: float,
                             seed: int, logger
                             ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Estratifica a nivel paciente y expande a imagenes.

    Paso 1: colapsar imagenes -> pacientes (un row por paciente).
    Paso 2: StratifiedKFold para separar test del resto.
    Paso 3: StratifiedKFold sobre train+val para separar val.
    Paso 4: expandir los conjuntos de pacientes a todas sus imagenes.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "Los ratios deben sumar 1.0"

    patient_df = collapse_to_patient_level(df)
    logger.info("Colapsado a %d pacientes unicos (%d positivos, %d negativos)",
                len(patient_df),
                int((patient_df['patient_label'] == 1).sum()),
                int((patient_df['patient_label'] == 0).sum()))

    # --- Paso 1: separar test ---
    n_splits_test = max(2, round(1.0 / test_ratio))
    skf_test = StratifiedKFold(n_splits=n_splits_test, shuffle=True,
                               random_state=seed)

    X_pat = patient_df['Patient ID'].to_numpy()
    y_pat = patient_df['patient_label'].to_numpy()

    train_val_idx, test_idx = next(skf_test.split(X_pat, y_pat))

    test_patients = set(X_pat[test_idx])
    train_val_patient_df = patient_df.iloc[train_val_idx].reset_index(drop=True)

    # --- Paso 2: separar val de train+val ---
    val_ratio_rel = val_ratio / (train_ratio + val_ratio)
    n_splits_val = max(2, round(1.0 / val_ratio_rel))
    skf_val = StratifiedKFold(n_splits=n_splits_val, shuffle=True,
                              random_state=seed)

    X_tv = train_val_patient_df['Patient ID'].to_numpy()
    y_tv = train_val_patient_df['patient_label'].to_numpy()

    train_idx, val_idx = next(skf_val.split(X_tv, y_tv))

    train_patients = set(X_tv[train_idx])
    val_patients = set(X_tv[val_idx])

    logger.info("Split a nivel paciente: train=%d, val=%d, test=%d pacientes",
                len(train_patients), len(val_patients), len(test_patients))

    # --- Paso 3: expandir a imagenes ---
    df_train = df[df['Patient ID'].isin(train_patients)].reset_index(drop=True)
    df_val = df[df['Patient ID'].isin(val_patients)].reset_index(drop=True)
    df_test = df[df['Patient ID'].isin(test_patients)].reset_index(drop=True)

    return df_train, df_val, df_test


# ---------------------------------------------------------------------------
# Reportes
# ---------------------------------------------------------------------------
def report_split(name: str, df: pd.DataFrame, logger) -> None:
    n = len(df)
    n_pos = int((df['label'] == 1).sum())
    n_neg = int((df['label'] == 0).sum())
    n_patients = df['Patient ID'].nunique()
    pct_pos = (n_pos / n * 100) if n else 0.0

    logger.info(
        "%-5s: %4d imgs | %3d pacientes | pos=%3d (%.1f%%) | neg=%3d",
        name.upper(), n, n_patients, n_pos, pct_pos, n_neg,
    )


def check_no_leakage(train: pd.DataFrame, val: pd.DataFrame,
                     test: pd.DataFrame, logger) -> None:
    p_train = set(train['Patient ID'])
    p_val = set(val['Patient ID'])
    p_test = set(test['Patient ID'])

    overlaps = {
        'train-val': p_train & p_val,
        'train-test': p_train & p_test,
        'val-test': p_val & p_test,
    }
    for name, overlap in overlaps.items():
        if overlap:
            logger.error("LEAKAGE en %s: %d pacientes compartidos", name, len(overlap))
            raise AssertionError(f"Data leakage detectado en {name}")

    logger.info("Sin leakage: pacientes disjuntos entre splits.")


def check_stratification(train: pd.DataFrame, val: pd.DataFrame,
                         test: pd.DataFrame, logger,
                         max_deviation: float = 10.0) -> None:
    combined = pd.concat([train, val, test], ignore_index=True)
    global_pct = (combined['label'] == 1).mean() * 100
    logger.info("Proporcion global de positivos: %.1f%%", global_pct)

    all_ok = True
    for name, df in [("train", train), ("val", val), ("test", test)]:
        pct = (df['label'] == 1).mean() * 100
        deviation = abs(pct - global_pct)
        if deviation > max_deviation:
            logger.warning(
                "%s tiene %.1f%% positivos (desviacion %.1fpp del global).",
                name, pct, deviation,
            )
            all_ok = False

    if all_ok:
        logger.info("Estratificacion OK: las 3 proporciones estan dentro de "
                    "%.0fpp del global.", max_deviation)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "baseline.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])
    log_file = Path(cfg.paths['logs']) / "prepare_splits.log"
    logger = get_logger("prepare_splits", log_file=log_file)

    logger.info("=" * 60)
    logger.info("PREPARE SPLITS v3 - Patient-level Stratified Split")
    logger.info("=" * 60)

    nih_dir = Path(cfg.paths['data_raw'])
    processed_dir = Path(cfg.paths['data_processed'])
    processed_dir.mkdir(parents=True, exist_ok=True)

    subset_csv = nih_dir / "Data_Entry_subset_local.csv"
    if not subset_csv.exists():
        raise FileNotFoundError(f"No se encontro {subset_csv}.")

    # 1. Cargar
    df = pd.read_csv(subset_csv)
    logger.info("CSV cargado: %d filas", len(df))

    # 2. Filtrar vistas
    df = filter_views(df, cfg.dataset['views'])
    logger.info("Tras filtrar vistas %s: %d filas", cfg.dataset['views'], len(df))

    # 3. Construir labels
    df = build_binary_labels(
        df,
        positive_class=cfg.dataset['positive_class'],
        negative_class=cfg.dataset['negative_class'],
    )
    n_before = len(df)
    df = df.dropna(subset=['label']).copy()
    df['label'] = df['label'].astype(int)
    logger.info(
        "Tras construir labels: %d filas (descartadas %d ambiguas)",
        len(df), n_before - len(df),
    )
    n_pos_img = int((df['label'] == 1).sum())
    n_neg_img = int((df['label'] == 0).sum())
    logger.info("  Imagenes positivas: %d | negativas: %d", n_pos_img, n_neg_img)

    # 4. Balancear a nivel paciente
    ratio = cfg.get('class_ratio', 2.0)
    df_balanced = undersample_negatives(df, ratio=ratio, seed=cfg.project['seed'])
    logger.info("Tras balancear a nivel paciente (ratio %.1f:1): %d imagenes, %d pacientes",
                ratio, len(df_balanced), df_balanced['Patient ID'].nunique())

    # 5. Split estratificado a nivel paciente
    train_df, val_df, test_df = stratified_patient_split(
        df_balanced,
        train_ratio=cfg.splits['train_ratio'],
        val_ratio=cfg.splits['val_ratio'],
        test_ratio=cfg.splits['test_ratio'],
        seed=cfg.project['seed'],
        logger=logger,
    )

    # 6. Reportar y validar
    logger.info("-" * 60)
    report_split("train", train_df, logger)
    report_split("val", val_df, logger)
    report_split("test", test_df, logger)
    logger.info("-" * 60)
    check_no_leakage(train_df, val_df, test_df, logger)
    check_stratification(train_df, val_df, test_df, logger)

    # 7. Guardar
    columns_to_save = ['Image Index', 'Finding Labels', 'Patient ID',
                       'View Position', 'label']
    train_df[columns_to_save].to_csv(processed_dir / "train.csv", index=False)
    val_df[columns_to_save].to_csv(processed_dir / "val.csv", index=False)
    test_df[columns_to_save].to_csv(processed_dir / "test.csv", index=False)

    logger.info("Splits guardados en: %s", processed_dir)
    logger.info("  train.csv: %d filas", len(train_df))
    logger.info("  val.csv:   %d filas", len(val_df))
    logger.info("  test.csv:  %d filas", len(test_df))
    logger.info("=" * 60)
    logger.info("PREPARE SPLITS v3 OK")


if __name__ == "__main__":
    main()
'''

script_path = f'{CODE_DIR}/scripts/prepare_splits.py'
Path(script_path).write_text(script_content, encoding='utf-8')
print(f'OK -> {script_path} ({len(script_content):,} chars)')


## 3. Ejecutar el script v3

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits.py'],
    capture_output=True, text=True
)
print('STDOUT:')
print(result.stdout)
print()
print('STDERR (log):')
print(result.stderr)


## 4. Verificar resultados finales

In [ ]:
import pandas as pd

processed = f'{PROJECT_ROOT}/data/processed'

print('RESUMEN FINAL DE SPLITS (v3):')
print('=' * 70)
total_pos = 0
total_neg = 0

for name in ['train', 'val', 'test']:
    df = pd.read_csv(f'{processed}/{name}.csv')
    n_pos = (df['label'] == 1).sum()
    n_neg = (df['label'] == 0).sum()
    n_pat = df['Patient ID'].nunique()
    pct = n_pos / len(df) * 100
    total_pos += n_pos
    total_neg += n_neg
    print(f'{name:5s}: {len(df):4d} imgs | {n_pat:3d} pacientes | '
          f'pos={n_pos:3d} ({pct:.1f}%) | neg={n_neg:3d}')

total = total_pos + total_neg
global_pct = total_pos / total * 100 if total else 0
print('-' * 70)
print(f'GLOBAL: {total} imgs | pos={total_pos} ({global_pct:.1f}%) | neg={total_neg}')
print()
print('COMPARATIVA:')
print('  v1 (random by patient):        train=34.3%, val=41.7%, test=19.8%')
print('  v2 (StratifiedGroupKFold):     train=35.8%, val=34.7%, test=12.1%  ← peor')
print('  v3 (Stratified patient-level): ← este resultado')


## 5. Siguiente paso

Si las tres proporciones de positivos estan ahora entre ~28% y ~38%
(diferencia < 10pp entre splits), vuelve al chat para recibir los dos archivos del
pipeline de datos:

1. `src/datasets/transforms.py`
2. `src/datasets/nih.py`
